In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("amazon_cleaned.csv")

print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nDuplicate Product IDs:")
print(df["product_id"].duplicated().sum())

print("\nMissing Values in Ranking Features:")
print(
    df[
        [
            "product_id",
            "rating",
            "rating_count",
            "discounted_price",
            "discount_percentage"
        ]
    ].isnull().sum()
)

Dataset Shape: (1348, 18)

Column Names:
['product_id', 'product_name', 'category', 'discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count', 'about_product', 'user_id', 'user_name', 'review_id', 'review_title', 'review_content', 'img_link', 'product_link', 'Cluster', 'Hierarchical_Cluster']

Duplicate Product IDs:
0

Missing Values in Ranking Features:
product_id             0
rating                 0
rating_count           0
discounted_price       0
discount_percentage    0
dtype: int64


In [2]:
# Remove duplicate product IDs
df = df.drop_duplicates(
    subset="product_id",
    keep="first"
).copy()

# Check feature ranges
features = [
    "rating",
    "rating_count",
    "discounted_price",
    "discount_percentage"
]

print("Products after removing duplicates:", len(df))

print("\nRanking Feature Statistics:")
display(df[features].describe().round(2))

Products after removing duplicates: 1348

Ranking Feature Statistics:


,rating,rating_count,discounted_price,discount_percentage
count,1348.00,1348.00,1348.00,1348.00
mean,4.09,17656.85,3310.27,46.66
std,0.30,42158.84,7180.90,21.61
min,2.00,2.00,39.00,0.00
25%,3.90,1107.50,349.00,31.00
50%,4.10,4740.00,899.00,49.00
75%,4.30,16051.50,2184.00,62.00
max,5.00,426973.00,77990.00,94.00


In [3]:
#i am using bayesian weighted rating
# Global average rating
C = df["rating"].mean()

# Median rating count as reliability benchmark
m = df["rating_count"].median()

# Bayesian weighted rating
df["weighted_rating"] = (
    (df["rating_count"] / (df["rating_count"] + m))
    * df["rating"]
    +
    (m / (df["rating_count"] + m))
    * C
)

print("Global Average Rating:", round(C, 4))
print("Reliability Benchmark:", m)

print("\nWeighted Rating Statistics:")
display(df["weighted_rating"].describe().round(4))

print("\nTop 10 Products by Weighted Rating:")
display(
    df.nlargest(10, "weighted_rating")[
        [
            "product_name",
            "rating",
            "rating_count",
            "weighted_rating"
        ]
    ]
)

Global Average Rating: 4.092
Reliability Benchmark: 4740.0

Weighted Rating Statistics:


count    1348.0000
mean        4.1128
std         0.1340
min         3.5316
25%         4.0566
50%         4.0976
75%         4.1824
max         4.7427
Name: weighted_rating, dtype: float64


Top 10 Products by Weighted Rating:


,product_name,rating,rating_count,weighted_rating
1029,Swiffer Instant Electric Water Heater Faucet T...,4.8,53803.0,4.742675
741,Redgear MP35 Speed-Type Gaming Mousepad (Black...,4.6,33434.0,4.536921
434,Spigen EZ Fit Tempered Glass Screen Protector ...,4.6,26603.0,4.523173
777,SanDisk Extreme SD UHS I 64GB Card for 4K Vide...,4.5,205052.0,4.490781
40,AmazonBasics USB 2.0 Cable - A-Male to B-Male ...,4.5,107687.0,4.482798
732,Crucial BX500 240GB 3D NAND SATA 6.35 cm (2.5-...,4.5,92925.0,4.480198
212,AmazonBasics USB 2.0 Extension Cable for Perso...,4.5,74977.0,4.475739
30,AmazonBasics USB 2.0 - A-Male to A-Female Exte...,4.5,74976.0,4.475739
444,Spigen EZ Fit Tempered Glass Screen Protector ...,4.7,7779.0,4.469792
663,"Logitech M235 Wireless Mouse, 1000 DPI Optical...",4.5,54405.0,4.467301


In [5]:
from sklearn.preprocessing import MinMaxScaler

# Log transformation to reduce rating-count skew
df["log_rating_count"] = np.log1p(df["rating_count"])

# Lower price = higher affordability score
df["price_affordability"] = 1 / df["discounted_price"]

features_to_normalize = [
    "weighted_rating",
    "log_rating_count",
    "price_affordability",
    "discount_percentage"
]

scaler = MinMaxScaler()

normalized_columns = [
    "normal_rating",
    "normal_popularity",
    "normal_price",
    "normal_discount"
]

df[normalized_columns] = scaler.fit_transform(
    df[features_to_normalize]
)

display(df[normalized_columns].describe().round(3))

,normal_rating,normal_popularity,normal_price,normal_discount
count,1348.000,1348.000,1348.000,1348.000
mean,0.480,0.602,0.082,0.496
std,0.111,0.172,0.106,0.230
min,0.000,0.000,0.000,0.000
25%,0.433,0.498,0.017,0.330
50%,0.467,0.621,0.043,0.521
75%,0.537,0.724,0.111,0.660
max,1.000,1.000,1.000,1.000


In [6]:
#assign weights & calculating ranking score
# Initial feature weights
weights = {
    "normal_rating": 0.40,
    "normal_popularity": 0.30,
    "normal_price": 0.15,
    "normal_discount": 0.15
}

# Verify weights add up to 1
print("Total Weight:", sum(weights.values()))

# Calculate final ranking score
df["ranking_score"] = (
    df["normal_rating"] * weights["normal_rating"]
    + df["normal_popularity"] * weights["normal_popularity"]
    + df["normal_price"] * weights["normal_price"]
    + df["normal_discount"] * weights["normal_discount"]
)

# Rank products
df = df.sort_values(
    by="ranking_score",
    ascending=False
).copy()

df["rank"] = range(1, len(df) + 1)

# Display top 10 products
top_10 = df[
    [
        "rank",
        "product_name",
        "rating",
        "rating_count",
        "discounted_price",
        "discount_percentage",
        "weighted_rating",
        "ranking_score"
    ]
].head(10)

display(top_10.round(4))

Total Weight: 1.0


,rank,product_name,rating,rating_count,discounted_price,discount_percentage,weighted_rating,ranking_score
65,1,"Amazon Basics High-Speed HDMI Cable, 6 Feet (2...",4.4,426973.0,309.0,78,4.3966,0.7290
12,2,AmazonBasics Flexible Premium HDMI Cable (Blac...,4.4,426973.0,219.0,69,4.3966,0.7225
40,3,AmazonBasics USB 2.0 Cable - A-Male to B-Male ...,4.5,107687.0,209.0,70,4.4828,0.7190
30,4,AmazonBasics USB 2.0 - A-Male to A-Female Exte...,4.5,74976.0,199.0,73,4.4757,0.7137
1029,5,Swiffer Instant Electric Water Heater Faucet T...,4.8,53803.0,1439.0,28,4.7427,0.6963
212,6,AmazonBasics USB 2.0 Extension Cable for Perso...,4.5,74977.0,299.0,63,4.4757,0.6879
457,7,ELV Aluminum Adjustable Mobile Phone Foldable ...,4.5,28978.0,269.0,82,4.4426,0.6854
777,8,SanDisk Extreme SD UHS I 64GB Card for 4K Vide...,4.5,205052.0,939.0,48,4.4908,0.6810
551,9,Elv Aluminium Adjustable Mobile Phone Foldable...,4.5,28978.0,314.0,79,4.4426,0.6775
396,10,STRIFF PS2_01 Multi Angle Mobile/Tablet Tablet...,4.3,42641.0,99.0,80,4.2792,0.6754


In [7]:
#weight sensitivity analysis
# Alternative weights
new_weights = {
    "normal_rating": 0.45,
    "normal_popularity": 0.25,
    "normal_price": 0.15,
    "normal_discount": 0.15
}

# Calculate alternative ranking score
df["new_ranking_score"] = (
    df["normal_rating"] * new_weights["normal_rating"]
    + df["normal_popularity"] * new_weights["normal_popularity"]
    + df["normal_price"] * new_weights["normal_price"]
    + df["normal_discount"] * new_weights["normal_discount"]
)

# Generate alternative ranks
df["new_rank"] = (
    df["new_ranking_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# Compare Top 10 product IDs
original_top10 = set(
    df.nsmallest(10, "rank")["product_id"]
)

new_top10 = set(
    df.nsmallest(10, "new_rank")["product_id"]
)

common_products = original_top10.intersection(new_top10)

print("Original Top 10:", len(original_top10))
print("New Top 10:", len(new_top10))
print("Common Products:", len(common_products))
print("Top 10 Stability:", f"{len(common_products) / 10 * 100:.1f}%")

# Display new Top 10
display(
    df.nsmallest(10, "new_rank")[
        [
            "new_rank",
            "rank",
            "product_name",
            "ranking_score",
            "new_ranking_score"
        ]
    ].round(4)
)

Original Top 10: 10
New Top 10: 10
Common Products: 9
Top 10 Stability: 90.0%


,new_rank,rank,product_name,ranking_score,new_ranking_score
65,1,1,"Amazon Basics High-Speed HDMI Cable, 6 Feet (2...",0.7290,0.7147
40,2,3,AmazonBasics USB 2.0 Cable - A-Male to B-Male ...,0.7190,0.7140
30,3,4,AmazonBasics USB 2.0 - A-Male to A-Female Exte...,0.7137,0.7100
12,4,2,AmazonBasics Flexible Premium HDMI Cable (Blac...,0.7225,0.7082
1029,5,5,Swiffer Instant Electric Water Heater Faucet T...,0.6963,0.7050
457,6,7,ELV Aluminum Adjustable Mobile Phone Foldable ...,0.6854,0.6844
212,7,6,AmazonBasics USB 2.0 Extension Cable for Perso...,0.6879,0.6842
551,8,9,Elv Aluminium Adjustable Mobile Phone Foldable...,0.6775,0.6765
777,9,8,SanDisk Extreme SD UHS I 64GB Card for 4K Vide...,0.6810,0.6737
434,10,11,Spigen EZ Fit Tempered Glass Screen Protector ...,0.6684,0.6711


In [8]:
# Compare original and new rankings
rank_comparison = df[
    [
        "product_id",
        "product_name",
        "rank",
        "new_rank",
        "ranking_score",
        "new_ranking_score"
    ]
].copy()

# Calculate rank movement
rank_comparison["rank_change"] = (
    rank_comparison["rank"] - rank_comparison["new_rank"]
)

# Show products appearing in either Top 10
top10_comparison = rank_comparison[
    (rank_comparison["rank"] <= 10) |
    (rank_comparison["new_rank"] <= 10)
].sort_values("rank")

display(top10_comparison.round(4))

# Spearman rank correlation across all products
from scipy.stats import spearmanr

rho, p_value = spearmanr(
    df["rank"],
    df["new_rank"]
)

print("Spearman Rank Correlation:", round(rho, 4))
print("P-value:", p_value)

print(
    "Average Absolute Rank Change (Top-10 union):",
    round(top10_comparison["rank_change"].abs().mean(), 2)
)

,product_id,product_name,rank,new_rank,ranking_score,new_ranking_score,rank_change
65,B014I8SX4Y,"Amazon Basics High-Speed HDMI Cable, 6 Feet (2...",1,1,0.7290,0.7147,0
12,B07KSMBL2H,AmazonBasics Flexible Premium HDMI Cable (Blac...,2,4,0.7225,0.7082,-2
40,B00NH11KIK,AmazonBasics USB 2.0 Cable - A-Male to B-Male ...,3,2,0.7190,0.7140,1
30,B00NH11PEY,AmazonBasics USB 2.0 - A-Male to A-Female Exte...,4,3,0.7137,0.7100,1
1029,B0BR4F878Q,Swiffer Instant Electric Water Heater Faucet T...,5,5,0.6963,0.7050,0
212,B00NH13Q8W,AmazonBasics USB 2.0 Extension Cable for Perso...,6,7,0.6879,0.6842,-1
457,B07Q4QV1DL,ELV Aluminum Adjustable Mobile Phone Foldable ...,7,6,0.6854,0.6844,1
777,B09X7DY7Q4,SanDisk Extreme SD UHS I 64GB Card for 4K Vide...,8,9,0.6810,0.6737,-1
551,B088ZFJY82,Elv Aluminium Adjustable Mobile Phone Foldable...,9,8,0.6775,0.6765,1
396,B07GXHC691,STRIFF PS2_01 Multi Angle Mobile/Tablet Tablet...,10,11,0.6754,0.6659,-1


Spearman Rank Correlation: 0.9924
P-value: 0.0
Average Absolute Rank Change (Top-10 union): 0.91


In [9]:
# Prepare final ranking dataset
ranking_export = df[
    [
        "product_id",
        "product_name",
        "category",
        "rating",
        "rating_count",
        "discounted_price",
        "discount_percentage",
        "weighted_rating",
        "ranking_score",
        "rank",
        "new_ranking_score",
        "new_rank"
    ]
].copy()

# Sort by original ranking
ranking_export = ranking_export.sort_values("rank")

# Export CSV
ranking_export.to_csv(
    "task_4_product_ranking.csv",
    index=False
)

print("CSV exported successfully!")
print("Total Products:", len(ranking_export))
print("Total Columns:", len(ranking_export.columns))

print("\nFinal Top 10:")
display(ranking_export.head(10))

CSV exported successfully!
Total Products: 1348
Total Columns: 12

Final Top 10:


,product_id,product_name,category,rating,rating_count,discounted_price,discount_percentage,weighted_rating,ranking_score,rank,new_ranking_score,new_rank
65,B014I8SX4Y,"Amazon Basics High-Speed HDMI Cable, 6 Feet (2...","Electronics|HomeTheater,TV&Video|Accessories|C...",4.4,426973.0,309.0,78,4.396618,0.729035,1,0.714748,1
12,B07KSMBL2H,AmazonBasics Flexible Premium HDMI Cable (Blac...,"Electronics|HomeTheater,TV&Video|Accessories|C...",4.4,426973.0,219.0,69,4.396618,0.722458,2,0.708170,4
40,B00NH11KIK,AmazonBasics USB 2.0 Cable - A-Male to B-Male ...,Computers&Accessories|Accessories&Peripherals|...,4.5,107687.0,209.0,70,4.482798,0.718970,3,0.714045,2
30,B00NH11PEY,AmazonBasics USB 2.0 - A-Male to A-Female Exte...,Computers&Accessories|Accessories&Peripherals|...,4.5,74976.0,199.0,73,4.475739,0.713679,4,0.709989,3
1029,B0BR4F878Q,Swiffer Instant Electric Water Heater Faucet T...,"Home&Kitchen|Heating,Cooling&AirQuality|WaterH...",4.8,53803.0,1439.0,28,4.742675,0.696303,5,0.705032,5
212,B00NH13Q8W,AmazonBasics USB 2.0 Extension Cable for Perso...,Computers&Accessories|Accessories&Peripherals|...,4.5,74977.0,299.0,63,4.475739,0.687886,6,0.684195,7
457,B07Q4QV1DL,ELV Aluminum Adjustable Mobile Phone Foldable ...,Electronics|Mobiles&Accessories|MobileAccessor...,4.5,28978.0,269.0,82,4.442643,0.685422,7,0.684370,6
777,B09X7DY7Q4,SanDisk Extreme SD UHS I 64GB Card for 4K Vide...,Electronics|Accessories|MemoryCards|MicroSD,4.5,205052.0,939.0,48,4.490781,0.681012,8,0.673703,9
551,B088ZFJY82,Elv Aluminium Adjustable Mobile Phone Foldable...,Electronics|Mobiles&Accessories|MobileAccessor...,4.5,28978.0,314.0,79,4.442643,0.677517,9,0.676465,8
396,B07GXHC691,STRIFF PS2_01 Multi Angle Mobile/Tablet Tablet...,Electronics|Mobiles&Accessories|MobileAccessor...,4.3,42641.0,99.0,80,4.279190,0.675372,10,0.665944,11


## Conclusion — Product Ranking Model

This project developed a weighted product ranking model for **1,348 Amazon products**, using customer ratings, rating counts, discounted prices, and discount percentages.

### Key Findings

1. **Data Preparation:** All 1,348 products had unique product IDs, and no missing values were found in the selected ranking features.

2. **Bayesian Weighted Rating:** Customer ratings were adjusted using a Bayesian weighted rating formula with a median rating count of 4,740 as the reliability benchmark. This reduced the unfair advantage of products with very few customer ratings.

3. **Feature Normalization:** Min-Max normalization was applied to adjusted ratings, log-transformed rating counts, inverse prices, and discount percentages to place all features on a comparable 0–1 scale.

4. **Ranking Weights:** The initial ranking model assigned 40% weight to adjusted rating, 30% to popularity, 15% to affordability, and 15% to discount percentage. These weights prioritize customer satisfaction and popularity while considering price and savings.

5. **Top-Ranked Product:** Amazon Basics High-Speed HDMI Cable achieved the highest initial ranking score of **0.7290**.

6. **Sensitivity Analysis:** After increasing the adjusted rating weight to 45% and reducing popularity to 25%, **9 out of the original Top 10 products remained in the Top 10**, representing 90% overlap.

7. **Ranking Stability:** Spearman's rank correlation between the original and modified rankings was **0.9924**, indicating very strong agreement. The original Rank 1 product retained its position.

### Limitations

The ranking model relies on ratings and rating counts as proxies for customer satisfaction and popularity because actual sales and profitability data are unavailable. The affordability feature may favor inexpensive products, and product categories have different typical price ranges. Ranking results also depend on the selected feature weights and normalization approach.

### Final Business Insight

The ranking model provides a systematic way to identify highly rated, popular, and affordable products with attractive discounts. The 90% Top-10 overlap suggests that the leading products are reasonably stable under the tested weight adjustment. However, businesses should validate the results using category-specific comparisons, actual sales data, and additional weight scenarios before making commercial decisions.